# ACADIA 2023 - Graph CSV Import Tutorial

## Working with Graphs and External Data in topologic_fast

This notebook is an adaptation of the ACADIA 2023 Graph CSV Import tutorial for the `topologic_fast` library.

### Topics Covered:
- Creating graphs from vertices and edges
- Simulating CSV data import
- Visualizing graphs with building geometry
- Graph analysis and metrics

### Note on CSV Import
The original topologicpy tutorial imports graphs from CSV files using `Graph.ByCSVPath()` and `DGL.GraphsByCSVPath()`. 
These features are not available in topologic_fast. This notebook demonstrates:
1. How to manually create graphs that would normally be imported
2. Alternative approaches for graph visualization
3. How to work with graph data in Python

In [ ]:
# Import libraries
import topologic_fast as tf
import plotly.graph_objects as go
import math
import numpy as np
import json

print("Done importing libraries")

## 1. Creating a Building CellComplex

First, let's create a building that would typically be imported from a BREP file.

In [ ]:
# NOTE: Topology.ByBREPPath is not available in topologic_fast
# We create a building programmatically instead

# Create a multi-story office building
building_cells = []
cell_labels = []  # Store labels for each cell (0-4 categories)

# Building parameters
floors = 8
rooms_per_floor_x = 4
rooms_per_floor_y = 3
room_width = 5.0
room_length = 6.0
floor_height = 3.5

for floor in range(floors):
    z = floor * floor_height
    
    for rx in range(rooms_per_floor_x):
        for ry in range(rooms_per_floor_y):
            x = rx * room_width
            y = ry * room_length
            
            cell = tf.Cell.Box(x, y, z, room_width, room_length, floor_height)
            building_cells.append(cell)
            
            # Assign labels based on position (simulating imported labels)
            # 0: Corner offices, 1: Side offices, 2: Interior, 3: Corridor, 4: Core
            is_corner = (rx == 0 or rx == rooms_per_floor_x - 1) and (ry == 0 or ry == rooms_per_floor_y - 1)
            is_edge = (rx == 0 or rx == rooms_per_floor_x - 1) or (ry == 0 or ry == rooms_per_floor_y - 1)
            is_center = (rx == rooms_per_floor_x // 2) and (ry == rooms_per_floor_y // 2)
            
            if is_center:
                label = 4  # Core
            elif is_corner:
                label = 0  # Corner offices
            elif is_edge:
                label = 1  # Side offices
            elif rx == rooms_per_floor_x // 2 or ry == rooms_per_floor_y // 2:
                label = 3  # Corridor
            else:
                label = 2  # Interior
            
            cell_labels.append(label)

# Create the CellComplex
cc = tf.CellComplex.ByCells(building_cells)

print(f"Building CellComplex created:")
print(f"  Total cells: {cc.NumCells()}")
print(f"  Total volume: {cc.Volume():.1f} m^3")
print(f"  Total area: {cc.Area():.1f} m^2")

# Print label distribution
label_names = ['Corner Office', 'Side Office', 'Interior', 'Corridor', 'Core']
print(f"\nLabel distribution:")
for i in range(5):
    count = cell_labels.count(i)
    print(f"  Label {i} ({label_names[i]}): {count} cells")

## 2. Creating a Graph from the Building

We create a dual graph where each room is represented by a vertex and connected rooms share an edge.

In [ ]:
# NOTE: In topologicpy, we would use Graph.ByCSVPath() to import from Grasshopper CSV
# Here we create the dual graph from the CellComplex

graph = tf.Graph.ByTopology(cc)

print(f"Graph created:")
print(f"  Vertices: {graph.Order()}")
print(f"  Edges: {graph.Size()}")
print(f"  Density: {graph.Density():.4f}")

In [ ]:
def show_graph_and_building(cellcomplex, graph, cell_labels, opacity=0.1):
    """Visualize building with overlaid graph"""
    fig = go.Figure()
    
    # Color map for labels
    colors = {
        0: '#FFD700',  # Gold - Corner
        1: '#90EE90',  # Light green - Side
        2: '#87CEEB',  # Sky blue - Interior
        3: '#D3D3D3',  # Light gray - Corridor
        4: '#FF6B6B'   # Red - Core
    }
    
    # Draw building faces with low opacity
    cells = cellcomplex.Cells()
    for i, cell in enumerate(cells):
        label = cell_labels[i] if i < len(cell_labels) else 0
        color = colors.get(label, '#808080')
        
        for face in cell.Faces():
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=opacity,
                    alphahull=0,
                    showlegend=False
                ))
    
    # Draw graph edges
    for edge in graph.Edges():
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='blue', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw graph vertices colored by label
    g_vertices = graph.Vertices()
    
    # Match graph vertices to cells by proximity
    vertex_labels = []
    for v in g_vertices:
        v_coords = v.Coordinates()
        min_dist = float('inf')
        closest_label = 0
        for i, cell in enumerate(cells):
            com = cell.CenterOfMass()
            dist = ((v_coords[0] - com[0])**2 + 
                   (v_coords[1] - com[1])**2 + 
                   (v_coords[2] - com[2])**2)
            if dist < min_dist:
                min_dist = dist
                closest_label = cell_labels[i] if i < len(cell_labels) else 0
        vertex_labels.append(closest_label)
    
    # Draw vertices by label group
    label_names = ['Corner Office', 'Side Office', 'Interior', 'Corridor', 'Core']
    for label in range(5):
        x, y, z = [], [], []
        for i, v in enumerate(g_vertices):
            if vertex_labels[i] == label:
                coords = v.Coordinates()
                x.append(coords[0])
                y.append(coords[1])
                z.append(coords[2])
        
        if x:
            fig.add_trace(go.Scatter3d(
                x=x, y=y, z=z,
                mode='markers',
                marker=dict(size=5, color=colors[label]),
                name=f'{label}: {label_names[label]}',
                hovertext=[f'Label: {label}' for _ in x],
                hoverinfo='text'
            ))
    
    fig.update_layout(
        title='Building with Dual Graph Overlay',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=1000, height=800,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

show_graph_and_building(cc, graph, cell_labels).show()

## 3. Simulating CSV Data Import

In topologicpy, you would import graph data from CSV files exported from Grasshopper. Here we show how to create similar data structures manually.

In [ ]:
# NOTE: Graph.ByCSVPath is not available in topologic_fast
# We simulate what the CSV import would provide

# Example of what the CSV files would contain:
# graphs.csv: graph_id, label (e.g., building energy performance)
# edges.csv: graph_id, source_id, target_id
# nodes.csv: graph_id, node_id, x, y, z, label

# Simulate graph data
class SimulatedGraphData:
    def __init__(self, graph, cell_labels):
        self.graph = graph
        self.vertices = graph.Vertices()
        self.edges = graph.Edges()
        self.labels = cell_labels
        
        # Simulate a "building performance" label (like energy efficiency)
        self.graph_label = 0.82  # e.g., 82% efficiency
        
    def to_dict(self):
        """Export graph to dictionary format (similar to CSV import result)"""
        nodes = []
        for i, v in enumerate(self.vertices):
            coords = v.Coordinates()
            nodes.append({
                'id': i,
                'x': coords[0],
                'y': coords[1],
                'z': coords[2],
                'label': self.labels[i] if i < len(self.labels) else 0
            })
        
        edges = []
        for edge in self.edges:
            edge_verts = edge.Vertices()
            if len(edge_verts) == 2:
                # Find vertex indices
                p1 = edge_verts[0].Coordinates()
                p2 = edge_verts[1].Coordinates()
                
                idx1 = self._find_vertex_index(p1)
                idx2 = self._find_vertex_index(p2)
                
                if idx1 is not None and idx2 is not None:
                    edges.append({'source': idx1, 'target': idx2})
        
        return {
            'graph_label': self.graph_label,
            'nodes': nodes,
            'edges': edges
        }
    
    def _find_vertex_index(self, coords):
        for i, v in enumerate(self.vertices):
            vc = v.Coordinates()
            if (abs(coords[0] - vc[0]) < 0.01 and 
                abs(coords[1] - vc[1]) < 0.01 and 
                abs(coords[2] - vc[2]) < 0.01):
                return i
        return None

# Create simulated data
sim_data = SimulatedGraphData(graph, cell_labels)
graph_dict = sim_data.to_dict()

print(f"Simulated graph data:")
print(f"  Graph label: {graph_dict['graph_label']}")
print(f"  Nodes: {len(graph_dict['nodes'])}")
print(f"  Edges: {len(graph_dict['edges'])}")
print(f"\nFirst 5 nodes:")
for node in graph_dict['nodes'][:5]:
    print(f"  {node}")

## 4. Graph Analysis

Analyze the graph properties and node connectivity.

In [ ]:
# Analyze graph properties
g_vertices = graph.Vertices()

# Calculate degree distribution
degrees = []
for v in g_vertices:
    degree = graph.VertexDegree(v)
    degrees.append(degree)

print(f"Graph Statistics:")
print(f"  Order (vertices): {graph.Order()}")
print(f"  Size (edges): {graph.Size()}")
print(f"  Density: {graph.Density():.4f}")
print(f"  Diameter: {graph.Diameter()}")
print(f"  Is Bipartite: {graph.IsBipartite()}")
print(f"  Is Connected: {graph.IsConnected()}")

print(f"\nDegree Statistics:")
print(f"  Min degree: {min(degrees)}")
print(f"  Max degree: {max(degrees)}")
print(f"  Mean degree: {sum(degrees)/len(degrees):.2f}")

In [ ]:
# Visualize degree distribution
import plotly.express as px

fig = px.histogram(
    x=degrees, 
    nbins=max(degrees) - min(degrees) + 1,
    title='Vertex Degree Distribution',
    labels={'x': 'Degree', 'y': 'Count'}
)
fig.update_layout(width=600, height=400)
fig.show()

## 5. Working with DGL (Deep Graph Library)

Note: DGL integration is not available in topologic_fast. This section shows what the equivalent operations would look like and how to prepare data for external DGL usage.

In [ ]:
# NOTE: DGL.GraphByTopologicGraph and DGL.DatasetByGraphs are not available in topologic_fast
# However, we can prepare data in a format suitable for DGL import

def prepare_dgl_data(graph, node_labels):
    """
    Prepare graph data in a format suitable for DGL import.
    
    In topologicpy, this would be:
    dgl_graph = DGL.GraphByTopologicGraph(graph, bidirectional=True, key="label", 
                                          categories=[0,1,2,3,4], node_attr_key='node_attr')
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Build edge index (source, target pairs)
    src_nodes = []
    dst_nodes = []
    
    # Create vertex coordinate to index mapping
    vertex_index = {}
    for i, v in enumerate(vertices):
        coords = tuple(v.Coordinates())
        vertex_index[coords] = i
    
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = tuple(edge_verts[0].Coordinates())
            p2 = tuple(edge_verts[1].Coordinates())
            
            # Round to avoid floating point issues
            p1 = tuple(round(c, 4) for c in p1)
            p2 = tuple(round(c, 4) for c in p2)
            
            # Find indices
            idx1 = None
            idx2 = None
            for key, idx in vertex_index.items():
                if all(abs(p1[i] - key[i]) < 0.01 for i in range(3)):
                    idx1 = idx
                if all(abs(p2[i] - key[i]) < 0.01 for i in range(3)):
                    idx2 = idx
            
            if idx1 is not None and idx2 is not None:
                # Bidirectional edges
                src_nodes.extend([idx1, idx2])
                dst_nodes.extend([idx2, idx1])
    
    # One-hot encode labels
    num_classes = 5
    node_features = []
    for i in range(len(vertices)):
        label = node_labels[i] if i < len(node_labels) else 0
        one_hot = [0.0] * num_classes
        one_hot[label] = 1.0
        node_features.append(one_hot)
    
    return {
        'num_nodes': len(vertices),
        'num_edges': len(src_nodes),
        'src_nodes': src_nodes,
        'dst_nodes': dst_nodes,
        'node_features': node_features,
        'node_labels': node_labels[:len(vertices)]
    }

# Prepare DGL-compatible data
dgl_data = prepare_dgl_data(graph, cell_labels)

print(f"DGL-compatible data prepared:")
print(f"  Nodes: {dgl_data['num_nodes']}")
print(f"  Edges (bidirectional): {dgl_data['num_edges']}")
print(f"  Feature dimensions: {len(dgl_data['node_features'][0])}")
print(f"\nExample node feature (one-hot encoded):")
print(f"  Node 0: {dgl_data['node_features'][0]}")

In [ ]:
# Show how to use this with DGL (if DGL is installed)
print("""
# To use with DGL, you would do:

import dgl
import torch

# Create DGL graph
g = dgl.graph((dgl_data['src_nodes'], dgl_data['dst_nodes']))

# Add node features
g.ndata['feat'] = torch.tensor(dgl_data['node_features'], dtype=torch.float32)
g.ndata['label'] = torch.tensor(dgl_data['node_labels'], dtype=torch.long)

print(g)
# Graph(num_nodes=96, num_edges=...,
#       ndata_schemes={'feat': Scheme(shape=(5,), dtype=torch.float32),
#                      'label': Scheme(shape=(), dtype=torch.int64)})
""")

## 6. Visualizing Graph with Labels

In [ ]:
def show_graph_labeled(graph, vertex_labels, title="Graph with Labels"):
    """Visualize graph with color-coded and labeled vertices"""
    fig = go.Figure()
    
    colors = {
        0: '#FFD700',  # Gold
        1: '#90EE90',  # Light green
        2: '#87CEEB',  # Sky blue
        3: '#D3D3D3',  # Light gray
        4: '#FF6B6B'   # Red
    }
    
    # Draw edges
    for edge in graph.Edges():
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='lightgrey', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    g_vertices = graph.Vertices()
    label_names = ['Corner', 'Side', 'Interior', 'Corridor', 'Core']
    
    for label in range(5):
        x, y, z, texts = [], [], [], []
        for i, v in enumerate(g_vertices):
            if i < len(vertex_labels) and vertex_labels[i] == label:
                coords = v.Coordinates()
                x.append(coords[0])
                y.append(coords[1])
                z.append(coords[2])
                texts.append(str(label))
        
        if x:
            fig.add_trace(go.Scatter3d(
                x=x, y=y, z=z,
                mode='markers+text',
                marker=dict(size=8, color=colors[label]),
                text=texts,
                textposition='top center',
                textfont=dict(size=10),
                name=f'{label}: {label_names[label]}',
                hovertext=[f'{label_names[label]}' for _ in x],
                hoverinfo='text'
            ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900, height=700
    )
    
    return fig

show_graph_labeled(graph, cell_labels, "Building Graph - Labeled by Room Type").show()

## 7. Adjacency Analysis

In [ ]:
# Analyze adjacency patterns
adj_matrix = graph.AdjacencyMatrix()

print(f"Adjacency Matrix shape: {len(adj_matrix)} x {len(adj_matrix[0]) if adj_matrix else 0}")

# Count adjacency by label pairs
label_names = ['Corner', 'Side', 'Interior', 'Corridor', 'Core']
label_adjacency = {}

for i in range(len(adj_matrix)):
    for j in range(i + 1, len(adj_matrix[i])):
        if adj_matrix[i][j] == 1:
            label_i = cell_labels[i] if i < len(cell_labels) else 0
            label_j = cell_labels[j] if j < len(cell_labels) else 0
            
            # Create sorted pair key
            pair = tuple(sorted([label_i, label_j]))
            label_adjacency[pair] = label_adjacency.get(pair, 0) + 1

print(f"\nAdjacency by room type pairs:")
for pair, count in sorted(label_adjacency.items(), key=lambda x: -x[1]):
    name1 = label_names[pair[0]]
    name2 = label_names[pair[1]]
    print(f"  {name1} <-> {name2}: {count}")

## Summary

In this tutorial, we learned:

1. **Building Creation** - Created a building CellComplex programmatically
2. **Graph Creation** - Generated dual graphs using `tf.Graph.ByTopology()`
3. **Data Simulation** - Simulated CSV import data structures
4. **Graph Analysis** - Analyzed graph properties and connectivity
5. **DGL Preparation** - Prepared data for external DGL usage
6. **Visualization** - Used Plotly for labeled graph visualization

### Features Not Available in topologic_fast

| topologicpy | topologic_fast | Alternative |
|-------------|----------------|-------------|
| `Topology.ByBREPPath(path)` | Not available | Create geometry programmatically |
| `Graph.ByCSVPath(...)` | Not available | Parse CSV manually and create graph |
| `DGL.GraphsByCSVPath(...)` | Not available | Use `prepare_dgl_data()` helper |
| `DGL.DatasetByCSVPath(...)` | Not available | Create DGL dataset externally |
| `DGL.GraphByTopologicGraph(...)` | Not available | Export adjacency and import to DGL |
| `Dictionary` operations | Not available | Store data in Python dicts |

### Working with External Data

When migrating from topologicpy to topologic_fast:

1. **BREP Import**: Create geometry using `Cell.Box()`, `Face.Rectangle()`, etc.
2. **CSV Import**: Parse CSV files with Python (pandas/csv) and create graphs manually
3. **DGL Integration**: Use the `prepare_dgl_data()` helper to format data for DGL
4. **Dictionary Data**: Store metadata in Python dictionaries alongside topologic objects